# Stage 2: Data Cleaning

Reloads the raw SVI/PLACES/NCHS CSVs, standardizes FIPS codes, cleans each
source, merges into one dataset, adds the census-region feature, and
imputes missing values.

**Requires:** raw CSVs in the Drive `data` folder (same as Stage 1).
**Produces:** `processed/county_data_cleaned.csv`

In [1]:
# IMPORT LIBRARIES
import os
import pandas as pd
import numpy as np

from google.colab import drive

In [2]:
# MOUNTING DRIVE AND LOADING RAW DATASETS - connect Colab to Drive folder
# Raw data files untouched, cleaning occurs on copies

drive.mount("/content/drive")

FOLDER_NAME = "data"
DATASET_PATH = f"/content/drive/MyDrive/{FOLDER_NAME}"

if os.path.exists(DATASET_PATH):
  print("✅ SUCCESS: Dataset folder located.")

  svi_file = "SVI_2022_US_county.csv"
  places_file = "PLACES__Local_Data_for_Better_Health,_County_Data,_2025_release_20260725.csv"
  nchs_file = "NCHSurb-rural-codes.csv"

  df_svi_raw = pd.read_csv(f"{DATASET_PATH}/{svi_file}")
  df_places_raw = pd.read_csv(f"{DATASET_PATH}/{places_file}", low_memory=False)
  df_nchs_raw = pd.read_csv(f"{DATASET_PATH}/{nchs_file}", encoding="latin-1")

  print ("✅ SUCCESS: All data sets loaded.")
  print(f"SVI:    {df_svi_raw.shape[0]} counties, {df_svi_raw.shape[1]} columns")
  print(f"PLACES: {df_places_raw.shape[0]} rows, {df_places_raw.shape[1]} columns")
  print(f"NCHS:   {df_nchs_raw.shape[0]} rows, {df_nchs_raw.shape[1]} columns")

else:
  print(f"❌ ERROR: {FOLDER_NAME} was not found in your MyDrive.")

Mounted at /content/drive
✅ SUCCESS: Dataset folder located.
✅ SUCCESS: All data sets loaded.
SVI:    3144 counties, 158 columns
PLACES: 229298 rows, 22 columns
NCHS:   3160 rows, 11 columns


In [3]:
# STANDARDIZING FIPS CODES FOR UNIFORMITY ACROSS ALL DATASETS
# FIPS = 5-digit county identifier
# Ensure all 3 FIPS dataset values identical for merge
# str.zfill adds leading zeros

df_svi_raw["FIPS"] = df_svi_raw["FIPS"].astype(str).str.zfill(5)
df_places_raw["FIPS"] = df_places_raw["LocationID"].astype(str).str.zfill(5)

# NCHS = no single FIPS column, combine STFIPS (state) + CTYFIPS (county)

df_nchs_raw["FIPS"] = (
    df_nchs_raw["STFIPS"].astype(str).str.zfill(2) +
    df_nchs_raw["CTYFIPS"].astype(str).str.zfill(3)
)

print("FIPS after standardization: ensure formatted with 5 digits ['01001',...]")
print("SVI: ", df_svi_raw["FIPS"].head(3).tolist())
print("PLACES: ", df_places_raw["FIPS"].head(3).tolist())
print("NCHS: ", df_nchs_raw["FIPS"].head(3).tolist())

FIPS after standardization: ensure formatted with 5 digits ['01001',...]
SVI:  ['01001', '01003', '01005']
PLACES:  ['05043', '05049', '05061']
NCHS:  ['01001', '01003', '01005']


In [4]:
# CLEAN SVI

svi = df_svi_raw.copy() # make copy to clean

# 1. Replace -999 values with NaN
svi = svi.replace(-999, np.nan)
print("Min RPL_THEMES after -999 replacement, if any (0.0 or NaN, not -999):")
print(svi["RPL_THEMES"].min())

# 2. Select evidence-based columns from literature findings in Silva et al. 2024
# /Ramphul et al. 2025
# EP_ = Estimate Percentage (raw %)
# RPL_ = Rank Percentile (0-1 scale, national ranking)

svi_cols = [
    # Identifiers for merging/labeling results
    "FIPS", "COUNTY", "ST_ABBR",

    # Overall SVI and four theme composites
    # RPL_THEMES = overall vulnerability - label engineering (Signal 1)
    # RPL_THEME4 = housing/transportation

    "RPL_THEMES","RPL_THEME1", "RPL_THEME2", "RPL_THEME3", "RPL_THEME4",

    # Individual features linked to literature findings
    "EP_NOVEH",   # % no vehicle - strongest individual SVI predictor
    "EP_AFAM",    # % Black/African American (OR = 1.69)
    "EP_UNINSUR", # % uninsured (OR = 1.90) (Medicaid/charity vs. private)
    "EP_LIMENG",  # % limited English (OR = 1.34)
    "EP_POV150",  # % below 150% poverty — both papers
    "EP_UNEMP",   # % unemployed — Silva
    "EP_NOHSDP",  # % no high school diploma — SES composite component
    "EP_HISP",    # % Hispanic/Latino — both papers
    "EP_MUNIT",   # % multiunit housing — housing instability proxy
    "EP_HBURD",   # % housing cost burden — Silva
    "EP_DISABL",  # % disability — household composition subscore
    "EP_MINRTY"   # % minority overall — composite minority measure
]

# Account for column title variations across annual SVI datasets
svi_cols = [c for c in svi_cols if c in svi.columns]
svi = svi[svi_cols].copy()

# Summary
print(f"\nSVI cleaned: {svi.shape[0]} counties, {svi.shape[1]} columns")
print("Remaining missing values:")
print(svi.isna().sum()[svi.isna().sum() > 0])

Min RPL_THEMES after -999 replacement, if any (0.0 or NaN, not -999):
0.0

SVI cleaned: 3144 counties, 20 columns
Remaining missing values:
Series([], dtype: int64)


In [5]:
# CLEAN PLACES

# Select evidence-based PLACES measures
places_measures = {
    "MAMMOUSE": "MAMMOGRAPHY",       # OB/GYN preventative care proxy replacing
                                     # CERVICAL, unavailable in PLACES 2025
    "CHECKUP": "ANNUAL_CHECKUP",     # healthcare system access proxy - Silva
    "OBESITY": "OBESITY_PREV",       # PMOS comorbidity - Silva, Ramphul, Neven
    "DIABETES": "DIABETES_PREV",     # PMOS comorbidity - Silva, Neven
    "DEPRESSION": "DEPRESSION_PREV", # PMOS psychosocial comorbidity - Neven
    "HIGHCHOL": "HIGH_CHOL_PREV",    # PMOS psychosocial comorbidity - Neven, Ramphul
                                     # 2% missing values, kept in
    "FOODINSECU": "FOOD_INSECURITY", # 23% missing values, SES/food access proxy — Ramphul
}

# 1. Filter rows to 7 evidence-based health measures
places_filtered = df_places_raw[
    (df_places_raw["MeasureId"].isin(places_measures.keys())) &
    (df_places_raw["Data_Value_Type"] == "Age-adjusted prevalence")
    ][["FIPS","MeasureId","Data_Value"]]

print("Rows found per measure (0 = wrong MeasureId string given in measures above):")
print(places_filtered["MeasureId"].value_counts())

# 2. Change from long (one row per county x measure)
# to wide format (one row per county, one col per measure)
places = places_filtered.pivot_table(
    index="FIPS",
    columns="MeasureId",
    values="Data_Value"
).reset_index()

# Rename cols to readable titles
places = places.rename(columns=places_measures)

# Summary
print(f"\nPLACES pivoted: {places.shape[0]} counties, {places.shape[1]} columns")
print("Remaining missing values (NaN):")
print(places.isna().sum())

Rows found per measure (0 = wrong MeasureId string given in measures above):
MeasureId
MAMMOUSE      3145
DEPRESSION    2958
OBESITY       2958
HIGHCHOL      2958
CHECKUP       2958
DIABETES      2958
FOODINSECU    2300
Name: count, dtype: int64

PLACES pivoted: 3144 counties, 8 columns
Remaining missing values (NaN):
MeasureId
FIPS                 0
ANNUAL_CHECKUP     187
DEPRESSION_PREV    187
DIABETES_PREV      187
FOOD_INSECURITY    844
HIGH_CHOL_PREV     187
MAMMOGRAPHY          0
OBESITY_PREV       187
dtype: int64


In [6]:
# CLEAN NCHS CODES
# Scale: 1 = large metro area | 6 = rural area (higher = inc rural)
# Select only FIPS and 2023 urban-rural code

nchs = df_nchs_raw[["FIPS","CODE2023"]].rename(
    columns={"CODE2023": "RURAL_CODE"}
).copy()

# Summary
print("Rural code distribution(1=most urban, 6=most rural):")
print(nchs["RURAL_CODE"].value_counts().sort_index())
print(f"Total # Codes: {nchs["RURAL_CODE"].count()}") # Ensure 3,144

Rural code distribution(1=most urban, 6=most rural):
RURAL_CODE
1.0      67
2.0     368
3.0     395
4.0     356
5.0     658
6.0    1300
Name: count, dtype: int64
Total # Codes: 3144


In [7]:
# MERGE ALL THREE DATASETS
# SVI (3,144 counties) as base
# PLACES and NCHS data matched by FIPS

df = svi.merge(places, on="FIPS", how="left")
df = df.merge(nchs, on="FIPS", how="left")

print(f"Merged shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Features included: {df.columns.tolist()}")

# Check for too few rows = FIPS mismatch betw datasets
# -> FIPS standardization issue
if df.shape[0] != 3144:
  print("⚠️ WARNING: merged file contains fewer counties than expected")
  print("SVI unique FIPS: ", svi["FIPS"].nunique())
  print("PLACES unique FIPS: ", places["FIPS"].nunique())
  print("NCHS unique FIPS: ", nchs["FIPS"].nunique())
else:
  print("✅ SUCCESS: all 3,144 counties and 25 features counted")

print("\nMissing values after merge:")
print(df.isna().sum()[df.isna().sum() > 0])

Merged shape: 3144 rows, 28 columns
Features included: ['FIPS', 'COUNTY', 'ST_ABBR', 'RPL_THEMES', 'RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3', 'RPL_THEME4', 'EP_NOVEH', 'EP_AFAM', 'EP_UNINSUR', 'EP_LIMENG', 'EP_POV150', 'EP_UNEMP', 'EP_NOHSDP', 'EP_HISP', 'EP_MUNIT', 'EP_HBURD', 'EP_DISABL', 'EP_MINRTY', 'ANNUAL_CHECKUP', 'DEPRESSION_PREV', 'DIABETES_PREV', 'FOOD_INSECURITY', 'HIGH_CHOL_PREV', 'MAMMOGRAPHY', 'OBESITY_PREV', 'RURAL_CODE']
✅ SUCCESS: all 3,144 counties and 25 features counted

Missing values after merge:
ANNUAL_CHECKUP     188
DEPRESSION_PREV    188
DIABETES_PREV      188
FOOD_INSECURITY    845
HIGH_CHOL_PREV     188
MAMMOGRAPHY          1
OBESITY_PREV       188
dtype: int64


In [8]:
# INVESTIGATING 188 MISSING PLACES VALUES
# MAMMOUSE has 6,290 rows in PLACES 2025 but other measures have 5,916
# After pivot: ~188 counties missing per measure — identify which and why

missing_counties = places[places["ANNUAL_CHECKUP"].isna()][["FIPS"]].copy()
missing_counties = missing_counties.merge(
    df[["FIPS", "COUNTY", "ST_ABBR", "RURAL_CODE", "EP_POV150"]],
    on="FIPS", how="left"
)

print(f"Counties missing PLACES measures (excl. MAMMOGRAPHY): {len(missing_counties)}")

# Check rural distribution — small/rural counties more likely suppressed by CDC
print("\nRural code distribution of missing counties (1=urban, 6=rural):")
print(missing_counties["RURAL_CODE"].value_counts().sort_index())

# Check state distribution — concentrated in specific states?
print("\nTop 10 states with missing counties:")
print(missing_counties["ST_ABBR"].value_counts().head(10))

# Compare poverty rate — are missing counties systematically poorer?
print(f"\nMean poverty rate (EP_POV150):")
print(f"  Missing counties: {missing_counties['EP_POV150'].mean():.2f}%")
print(f"  All counties:     {df['EP_POV150'].mean():.2f}%")

Counties missing PLACES measures (excl. MAMMOGRAPHY): 187

Rural code distribution of missing counties (1=urban, 6=rural):
RURAL_CODE
1.0     3
2.0    25
3.0    25
4.0    19
5.0    49
6.0    66
Name: count, dtype: int64

Top 10 states with missing counties:
ST_ABBR
KY    120
PA     67
Name: count, dtype: int64

Mean poverty rate (EP_POV150):
  Missing counties: 26.49%
  All counties:     23.95%


In [9]:
# ADDING CENSUS REGION AS A GEOGRAPHIC FEATURE
# RURAL_CODE is excluded in model training due to use as Signal 3 in label
# Region is different geographic feature - carries different signal than NCHS
# codes, supported by Visualization 3 (Southern/gulf states = higher SVI)

# State abbrievation: official US Census Bureau region
region_map = {
    'CT':'Northeast','ME':'Northeast','MA':'Northeast','NH':'Northeast',
    'RI':'Northeast','VT':'Northeast','NJ':'Northeast','NY':'Northeast',
    'PA':'Northeast',
    'IL':'Midwest','IN':'Midwest','MI':'Midwest','OH':'Midwest','WI':'Midwest',
    'IA':'Midwest','KS':'Midwest','MN':'Midwest','MO':'Midwest','NE':'Midwest',
    'ND':'Midwest','SD':'Midwest',
    'DE':'South','FL':'South','GA':'South','MD':'South','NC':'South',
    'SC':'South','VA':'South','DC':'South','WV':'South','AL':'South',
    'KY':'South','MS':'South','TN':'South','AR':'South','LA':'South',
    'OK':'South','TX':'South',
    'AZ':'West','CO':'West','ID':'West','MT':'West','NV':'West','NM':'West',
    'UT':'West','WY':'West','AK':'West','CA':'West','HI':'West','OR':'West',
    'WA':'West'
}

df['REGION'] = df['ST_ABBR'].map(region_map)

# Error check for missing states - should be empty
print("Unmapped states (should be empty):", df[df['REGION'].isna()]
 ['ST_ABBR'].unique())
# Ensure all regions filled
print("\nCounties per region:")
print(df['REGION'].value_counts())

# Encoding step, converts to int 0-1, creates one new col/val
# Drop first region alphabetically to prevent multicollinearity
df = pd.get_dummies(df, columns=['REGION'], drop_first=True, dtype=int)

# Midwest (first region, baseline (0,0,0)) should be dropped
print("\nNew columns added:",
 [c for c in df.columns if c.startswith('REGION_')])

Unmapped states (should be empty): []

Counties per region:
REGION
South        1422
Midwest      1055
West          449
Northeast     218
Name: count, dtype: int64

New columns added: ['REGION_Northeast', 'REGION_South', 'REGION_West']


In [10]:
# PRE-IMPUTATION BIAS CHECK
# Before filling missing values, check whether missing counties differ
# systematically from complete counties on key demographic indicators
# Systematic differences = imputation may introduce bias

# Columns with missing values (from merge output)
missing_cols = [c for c in df.columns if df[c].isna().any()]
print(f"Columns with missing values: {missing_cols}\n")

# For each missing column, compare key demographics of missing vs complete counties
check_cols = ["EP_POV150", "EP_AFAM", "EP_MINRTY", "RURAL_CODE"]

rows = []
for col in missing_cols:
    missing_mask = df[col].isna()
    for demo in check_cols:
        rows.append({
            "Missing Column": col,
            "Demographic":    demo,
            "Missing Mean":   df.loc[missing_mask,  demo].mean(),
            "Complete Mean":  df.loc[~missing_mask, demo].mean(),
            "N Missing":      missing_mask.sum(),
        })

bias_check = pd.DataFrame(rows).round(3)
print("Mean demographic values: missing vs complete counties")
print(bias_check.to_string(index=False))

Columns with missing values: ['ANNUAL_CHECKUP', 'DEPRESSION_PREV', 'DIABETES_PREV', 'FOOD_INSECURITY', 'HIGH_CHOL_PREV', 'MAMMOGRAPHY', 'OBESITY_PREV']

Mean demographic values: missing vs complete counties
 Missing Column Demographic  Missing Mean  Complete Mean  N Missing
 ANNUAL_CHECKUP   EP_POV150        26.391         23.800        188
 ANNUAL_CHECKUP     EP_AFAM         3.716          9.052        188
 ANNUAL_CHECKUP   EP_MINRTY        10.914         26.204        188
 ANNUAL_CHECKUP  RURAL_CODE         4.527          4.618        188
DEPRESSION_PREV   EP_POV150        26.391         23.800        188
DEPRESSION_PREV     EP_AFAM         3.716          9.052        188
DEPRESSION_PREV   EP_MINRTY        10.914         26.204        188
DEPRESSION_PREV  RURAL_CODE         4.527          4.618        188
  DIABETES_PREV   EP_POV150        26.391         23.800        188
  DIABETES_PREV     EP_AFAM         3.716          9.052        188
  DIABETES_PREV   EP_MINRTY        10.914    

In [11]:
# HANDLE MISSING VALUES

# Check for missing values
missing_pct = df.isna().mean().sort_values(ascending=False)
print("Missing value rates (columns with any missing values):")
print(missing_pct[missing_pct > 0].round(3))

# 1. Drop columns with > 30% missing, but check validate with literature
high_missing = missing_pct[missing_pct > 0.30].index.tolist()
if high_missing:
  print(f"\nDropping {len(high_missing)} high missing values column(s): {high_missing}")
  print(f"\n‼️ Please validate dropped columns to ensure proper exclusion")
  df = df.drop(columns=high_missing)
else:
  print("\nAll columns kept. No columns exceed 30% missing values.")

# 2. Fill missing values w/ column median to account for outliers
num_cols = df.select_dtypes(include=[np.number]).columns
filled_cols = [col for col in num_cols if df[col].isna().any()]
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

remaining = df.isna().sum().sum()
print(f"\nMissing values remaining: {remaining}")
if remaining == 0:
  print("✅ SUCCESS: Missing values cleaned from merged dataset.")
  print(f"Columns filled with medians: {filled_cols}")
else:
  print(f"⚠️ WARNING: {remaining} missing values still exist.")

Missing value rates (columns with any missing values):
FOOD_INSECURITY    0.269
OBESITY_PREV       0.060
ANNUAL_CHECKUP     0.060
DIABETES_PREV      0.060
HIGH_CHOL_PREV     0.060
DEPRESSION_PREV    0.060
MAMMOGRAPHY        0.000
dtype: float64

All columns kept. No columns exceed 30% missing values.

Missing values remaining: 0
✅ SUCCESS: Missing values cleaned from merged dataset.
Columns filled with medians: ['ANNUAL_CHECKUP', 'DEPRESSION_PREV', 'DIABETES_PREV', 'FOOD_INSECURITY', 'HIGH_CHOL_PREV', 'MAMMOGRAPHY', 'OBESITY_PREV']


In [12]:
# SAVE CLEANED + MERGED DATASET FOR DOWNSTREAM NOTEBOOKS
PROCESSED_DIR = f"{DATASET_PATH}/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)
df.to_csv(f"{PROCESSED_DIR}/county_data_cleaned.csv", index=False)

print(f"✅ SUCCESS: cleaned dataset saved to {PROCESSED_DIR}/county_data_cleaned.csv")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

✅ SUCCESS: cleaned dataset saved to /content/drive/MyDrive/data/processed/county_data_cleaned.csv
Shape: 3144 rows, 31 columns
